# 🎮 Tarea 2: API REST — Base de datos de videojuegos RAWG

En este cuaderno se consume la API de [RAWG](https://rawg.io/apidocs) para extraer, analizar y comparar datos de videojuegos usando Python.

**Alumno:** Tu Nombre Aquí  
**Fecha:** 2026  
**API utilizada:** RAWG Video Games Database API

## ⚙️ Configuración inicial

Se cargan las librerías necesarias y la API Key desde un archivo `.env` para no exponerla en el código.

In [ ]:
# Instalar dependencias si es necesario
# !pip install requests pandas python-dotenv

In [ ]:
import os
import requests
import pandas as pd
from pathlib import Path
from typing import Any
from dotenv import load_dotenv

# Busca el archivo .env en la carpeta actual o en carpetas padre
for p in [Path.cwd(), *Path.cwd().parents]:
    candidate = p / '.env'
    if candidate.exists():
        load_dotenv(candidate)
        break

# Lee la API Key desde .env  (crea un archivo .env con: RAWG_API_KEY=tu_clave_aqui)
API_KEY = os.getenv('RAWG_API_KEY')
if not API_KEY:
    raise ValueError('No se encontró RAWG_API_KEY en el archivo .env')

BASE_URL = 'https://api.rawg.io/api'

print('✅ Librerías cargadas correctamente')
print(f'🔑 API Key detectada: {API_KEY[:6]}...(oculta por seguridad)')

In [ ]:
# Cliente RAWG: centraliza los requests y lleva un contador automático
class RAWGClient:
    def __init__(self, api_key: str, timeout: int = 30):
        self.api_key = api_key
        self.timeout = timeout
        self._requests = 0

    def _get(self, endpoint: str, **params: Any) -> dict:
        """Realiza una petición GET a la API de RAWG."""
        url = f"{BASE_URL}/{endpoint}"
        query = {'key': self.api_key, **params}
        r = requests.get(url, params=query, timeout=self.timeout)
        self._requests += 1
        r.raise_for_status()
        return r.json()

    def games(self, **params: Any) -> dict:
        """Consulta el endpoint /games con los parámetros dados."""
        return self._get('games', **params)

    def game_detail(self, game_id: int) -> dict:
        """Obtiene el detalle de un juego por su ID."""
        return self._get(f'games/{game_id}')

    def resumen_requests(self) -> int:
        """Imprime y retorna el total de requests realizados."""
        print(f'📊 Total de requests realizados: {self._requests}')
        return self._requests

client = RAWGClient(API_KEY)
print('✅ Cliente RAWG listo')

---
## 🟢 Parte A — Exploración General (2 pts)

### A1 — Total de juegos registrados en RAWG

Consultamos el endpoint `/games` con `page_size=1` para obtener solo el campo `count`, que contiene el total de juegos en la base de datos.

In [ ]:
# A1: Conteo total de juegos en RAWG
resp_total = client.games(page_size=1)
total_games = resp_total.get('count', 0)

print(f'🎮 Total de juegos registrados en RAWG: {total_games:,}')

---
## 🔵 Parte B — Análisis de Categoría (2 pts)

### B1 — Top 5 juegos mejor valorados por Metacritic

Usamos `ordering=-metacritic` para ordenar de mayor a menor puntaje de crítica especializada.

In [ ]:
# B1: Top 5 por puntuación Metacritic
resp_top_meta = client.games(ordering='-metacritic', page_size=5)
top5_meta = pd.DataFrame(resp_top_meta['results'])[['name', 'rating', 'metacritic']]

print('🏆 B1) Top 5 juegos mejor valorados por Metacritic:')
display(top5_meta)

### B2 — Top 10 juegos en Steam (store_id = 1)

Filtramos por `stores=1` (Steam) y ordenamos por `rating` de usuarios.

In [ ]:
# B2: Top 10 en Steam ordenados por rating de usuarios
resp_steam = client.games(stores=1, ordering='-rating', page_size=10)
top10_steam = pd.DataFrame(resp_steam['results'])[['name', 'rating', 'metacritic']]

print('🛒 B2) Top 10 juegos en Steam:')
display(top10_steam)

---
## 🟡 Parte C — Comparaciones (3 pts)

### C1 — PC (platform_id=4) vs PS5 (platform_id=187)

Obtenemos el Top 5 de cada plataforma y calculamos el rating promedio para determinar cuál tiene los juegos mejor valorados.

In [ ]:
# C1: Comparativa PC vs PS5
pc_resp  = client.games(platforms=4,   ordering='-rating', page_size=5)
ps5_resp = client.games(platforms=187, ordering='-rating', page_size=5)

pc_top5  = pd.DataFrame(pc_resp['results'])[['name', 'rating', 'metacritic']]
ps5_top5 = pd.DataFrame(ps5_resp['results'])[['name', 'rating', 'metacritic']]

pc_avg  = pc_top5['rating'].mean()
ps5_avg = ps5_top5['rating'].mean()
best_platform = 'PC' if pc_avg > ps5_avg else 'PS5'

print(f'💻 Promedio rating PC:  {pc_avg:.3f}')
print(f'🎮 Promedio rating PS5: {ps5_avg:.3f}')
print(f'🏆 Plataforma con juegos mejor valorados: {best_platform}')

print('\nTop 5 — PC:')
display(pc_top5)
print('Top 5 — PS5:')
display(ps5_top5)

### C2 — Tabla comparativa de 3 juegos famosos

Buscamos 3 juegos icónicos y comparamos nombre, rating, Metacritic, géneros y plataformas.

In [ ]:
# C2: Tabla comparativa — cambia estos 3 juegos por los que tú elijas
famous_titles = ['God of War', 'Hades', 'Hollow Knight']

rows = []
for title in famous_titles:
    result = client.games(search=title, page_size=1).get('results', [])
    if not result:
        continue
    g = result[0]
    rows.append({
        'name':       g.get('name'),
        'rating':     g.get('rating'),
        'metacritic': g.get('metacritic'),
        'genres':     ', '.join(x['name'] for x in g.get('genres', [])),
        'platforms':  ', '.join(x['platform']['name'] for x in g.get('platforms', [])[:5]),
    })

c2_df = pd.DataFrame(rows)
print('🎯 C2) Comparativa de 3 juegos famosos:')
display(c2_df)

### C3 — Rating promedio por género

Consultamos el Top 5 de 4 géneros distintos, calculamos el rating promedio de cada uno y determinamos qué género produce los juegos mejor valorados.

In [ ]:
# C3: Comparativa de géneros — IDs: Action=4, RPG=5, Shooter=2, Strategy=10
genre_map = {'Action': 4, 'RPG': 5, 'Shooter': 2, 'Strategy': 10}

rows = []
for genre_name, genre_id in genre_map.items():
    result = client.games(genres=genre_id, ordering='-rating', page_size=5)
    gdf = pd.DataFrame(result['results'])[['name', 'rating']]
    rows.append({
        'genre':           genre_name,
        'avg_rating_top5': round(gdf['rating'].mean(), 3)
    })

c3_df = pd.DataFrame(rows).sort_values('avg_rating_top5', ascending=False).reset_index(drop=True)
best_genre = c3_df.iloc[0]['genre']

print(f'🏆 C3) Género con mejor rating promedio: {best_genre}')
display(c3_df)

### C4 — Comparativa por año de lanzamiento

Comparamos los mejores juegos de 3 años distintos según su Metacritic promedio.

In [ ]:
# C4: Comparativa por año — cambia estos años por los que tú elijas
years = [2017, 2019, 2022]

rows = []
for y in years:
    result = client.games(dates=f'{y}-01-01,{y}-12-31', ordering='-metacritic', page_size=20)
    ydf = pd.DataFrame(result['results'])[['name', 'metacritic']]
    rows.append({
        'year':                 y,
        'avg_metacritic_top20': round(ydf['metacritic'].dropna().mean(), 1)
    })

c4_df = pd.DataFrame(rows).sort_values('avg_metacritic_top20', ascending=False).reset_index(drop=True)
best_year = int(c4_df.iloc[0]['year'])

print(f'📅 C4) Año con mayor Metacritic promedio: {best_year}')
display(c4_df)

### C5 — Exportar Top 20 a CSV

Obtenemos los 20 mejores juegos de todos los tiempos por Metacritic y los exportamos a `output/top20_rawg.csv` con las columnas requeridas.

In [ ]:
# C5: Generar top20_rawg.csv en la carpeta output/
out_dir = Path('output')
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / 'top20_rawg.csv'

resp_top20 = client.games(ordering='-metacritic', page_size=20)

rows = []
for g in resp_top20['results']:
    rows.append({
        'name':         g.get('name'),
        'rating':       g.get('rating'),
        'metacritic':   g.get('metacritic'),
        'release_date': g.get('released'),
        'main_genre':   g['genres'][0]['name'] if g.get('genres') else None,
    })

top20_df = pd.DataFrame(rows)
top20_df.to_csv(out_csv, index=False, encoding='utf-8')

print(f'✅ CSV exportado en: {out_csv.resolve()}')
print(f'📄 Total filas: {len(top20_df)}')
print('\n📋 Primeras 5 filas:')
display(pd.read_csv(out_csv).head())

---
## 🔴 Parte D — Reflexiones y Conclusiones (3 pts)

### D1 — Conclusiones personales

---

#### 🔍 ¿Qué fue lo más interesante que encontraste en los datos?

Lo más interesante fue comprobar que **Metacritic y el rating de usuarios no siempre coinciden**.
Juegos clásicos como *The Legend of Zelda: Ocarina of Time* tienen Metacritic de 99,
pero su rating de usuarios es 4.38 sobre 5. En cambio, expansiones de *The Witcher 3*
en Steam alcanzan ratings de 4.80 con un Metacritic de apenas 92.
Esto revela que la crítica especializada y la comunidad de jugadores valoran
aspectos muy distintos en un videojuego.

#### 😮 ¿Qué género o plataforma te sorprendió más y por qué?

Me sorprendió que **RPG supere a Action en rating promedio**, siendo Action
el género más popular y masivo. Esperaba que la popularidad se tradujera en
mejores valoraciones, pero los RPGs generan experiencias más profundas que
los usuarios valoran más. En cuanto a plataformas, me sorprendió que **PC
supere a PS5** en promedio, ya que PS5 es una consola más reciente con
títulos exclusivos muy premiados. La clave parece estar en que PC tiene un
catálogo mucho más amplio con más títulos altamente valorados.

#### ❓ ¿Qué otra pregunta le harías a esta API si tuvieras más tiempo?

1. **¿Cómo evoluciona el rating promedio por género a lo largo de los años?**
   ¿Están mejorando los RPGs con el tiempo o siempre fue así?
2. **¿Qué desarrolladora acumula más juegos en el top 100 de todos los tiempos?**
   ¿Hay alguna que domine consistentemente las listas?
3. **¿Existe correlación entre el número de plataformas en que sale un juego
   y su rating?** ¿Los exclusivos son mejor valorados que los multiplataforma?

#### 📊 ¿Cuántas solicitudes utilizaste en total?
Ver celda de código a continuación con `client.resumen_requests()`.

---

In [ ]:
# D1: Total de requests utilizados durante toda la tarea
client.resumen_requests()